# Simple QQQ Direction Backtest (N=50, sequential, in-memory)

Fetch QQQ from 2025-10-01 (warmup — 50 trading days before 2026-01-02 reaches back into Oct 2025) → 2026-08-30, loop each trading day 2026-01-02 → 2026-08-30,
predict from the previous 50 trading days (excluding the current day), check the actual, store a datapoint.
No threads, no cache files, no CLI — everything stays in memory.

**Prereq:** `export TYPESAFE_API_KEY="..."` before starting the kernel.

Run top-to-bottom: `Kernel → Restart & Run All`.

In [ ]:
import os
if not os.environ.get('TYPESAFE_API_KEY', '').strip():
    raise RuntimeError('TYPESAFE_API_KEY env var is not set — export it, then restart the kernel.')

FETCH_START = '2025-10-01'  # warmup buffer so early-January targets have 50 prior trading days
START = '2026-01-02'
END = '2026-08-30'
N = 50  # previous trading days per prediction (current day excluded)

import datetime
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from predict_qqq_direction import build_state, predict_direction, validate_num_timepoints

validate_num_timepoints(N)
print(f'simple backtest: {START}..{END}  N={N}  warmup from {FETCH_START}')

## 1. Fetch QQQ closes (warmup included)

In [ ]:
import yfinance as yf

px = yf.download('QQQ', start=FETCH_START, end='2026-08-31', interval='1d', auto_adjust=True, progress=False)
if hasattr(px.columns, 'droplevel'):
    try:
        px.columns = px.columns.droplevel(1)
    except (ValueError, IndexError, KeyError, TypeError):
        pass
col = 'Close' if 'Close' in px.columns else 'Adj Close'
price = px[[col]].rename(columns={col: 'Close'}).dropna().copy()
price.index = pd.to_datetime(price.index).tz_localize(None)
closes = [float(v) for v in price['Close'].tolist()]
trading_days = [ts for ts in price.index.tolist()]
start_d = datetime.date.fromisoformat(START)
end_d = datetime.date.fromisoformat(END)
targets = [d for d in trading_days if start_d <= d.date() <= end_d]
assert all(trading_days.index(d) >= N for d in targets), 'not enough warmup history for 50 prior days'
print(f'closes: {len(closes)}  target days: {len(targets)}  first: {targets[0].date()}  last: {targets[-1].date()}')

## 2. The loop — previous 50 days → predict → check actual → store datapoint

In [ ]:

records = []
for i, d in enumerate(targets):
    j = trading_days.index(d)
    prior = closes[j - N - 1:j]  # 51 closes -> 50 log changes, current day excluded
    changes = [(cur - prev) / prev * 100.0 for prev, cur in zip(prior, prior[1:])]  # simple % changes
    try:
        result = predict_direction(build_state(changes, N))
        predicted = result['prediction']
        actual_ret = (closes[j] - closes[j - 1]) / closes[j - 1] * 100.0  # direct percent change
        actual = 'UP' if actual_ret > 0 else 'DOWN'
        records.append({'date': d, 'predicted': predicted, 'actual': actual,
                        'confidence': float(result['confidence']),
                        'actual_change_pct': round(actual_ret, 4),
                        'correct': predicted == actual})
    except Exception as exc:
        print(f'{d.date()}: skipped ({exc})')
    if (i + 1) % 25 == 0 or i + 1 == len(targets):
        print(f'{i + 1}/{len(targets)} days done')
print(f'stored datapoints: {len(records)}')

## 3. Analysis + chart

In [ ]:
df = pd.DataFrame(records).set_index('date').sort_index()
acc = df['correct'].mean()
tp = int(((df['predicted'] == 'UP') & (df['actual'] == 'UP')).sum())
fp = int(((df['predicted'] == 'UP') & (df['actual'] == 'DOWN')).sum())
tn = int(((df['predicted'] == 'DOWN') & (df['actual'] == 'DOWN')).sum())
fn = int(((df['predicted'] == 'DOWN') & (df['actual'] == 'UP')).sum())
print(f"accuracy: {acc:.4f} ({int(df['correct'].sum())}/{len(df)})")
print(f'confusion: UP/UP={tp}  UP/DOWN={fp}  DOWN/UP={fn}  DOWN/DOWN={tn}')

fig, ax = plt.subplots(figsize=(14, 5))
runs, s = [], 0
preds = df['predicted'].tolist()
dates = df.index.tolist()
for k in range(1, len(preds) + 1):
    if k == len(preds) or preds[k] != preds[s]:
        runs.append((dates[s], dates[k - 1] + pd.Timedelta(days=1), preds[s]))
        s = k
for d0, d1, lab in runs:
    ax.axvspan(d0, d1, color='green' if lab == 'UP' else 'red', alpha=0.3, linewidth=0)
ax.plot(price.index, price['Close'], color='#1f3a93', lw=1.5, zorder=3, label='QQQ Close')
ax.set_title(f'QQQ Price with Direction Predictions ({len(df)} days)', fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('Price ($)')
ax.legend(handles=[mpatches.Patch(color='green', alpha=0.3, label='Predicted UP'),
                   mpatches.Patch(color='red', alpha=0.3, label='Predicted DOWN'),
                   plt.Line2D([], [], color='#1f3a93', label='QQQ Close')], loc='upper left')
fig.tight_layout(); plt.show()